In [2]:
import pandas as pd 
import numpy as np
import scipy.stats as sci 


/Users/mienguyen/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Part I. PnL Explain: Long Call

Goal: explain the 1-day PnL of a long call using Greeks, then compare with the actual PnL from full revaluation.

| Symbol | Meaning |
|---|---|
| S | Spot price |
| K | Strike |
| T | Time to maturity (years) |
| r | Risk-free rate |
| sigma | Volatility |

### 1. Black-Scholes price and Greeks (call, computed at day 0)

In [3]:
# d1 = [ln(S/K) + (r + σ²/2)* T] / (σ√T)
# d2 = d1 − σ√T
# Call price = S * N(d1) − K * e^(-rT)* N(d2)
def bs_call_prices(S,K,T,r,sigma):
    d1 = (np.log(S/K) + ( r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    price = S * sci.norm.cdf(d1) - K * np.exp(-r*T) * sci.norm.cdf(d2)
    return price
bs_call_prices(100, 100, 30/365, 0.02, 0.20)

2.3683354165525543

In [4]:
# ATM call: Delta ≈ 0.5
def bs_call_delta(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    delta = sci.norm.cdf(d1)
    return delta
bs_call_delta (100,100,30/365, 0.02, 0.20)

0.5228621115361781

In [5]:
# Gamma = n(d1) / (S * σ * √T)
def bs_gamma(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    gamma = sci.norm.pdf(d1) / ( S * sigma * np.sqrt(T))
    return gamma
bs_gamma(100,100,30/365,0.02,0.20)



0.06946274895134844

In [6]:
# Vega per 1 vol point = S * n(d1) * √T / 100
def bs_vega(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    vega = S * sci.norm.pdf(d1) * np.sqrt(T)/100
    return vega
bs_vega(100,100,30/365,0.02,0.20)


0.11418534074194264

In [7]:
# Call Theta per day = [-S*n(d1)*σ/(2√T) - r*K*e^(-rT)*N(d2)] / 365
def bs_call_theta(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    theta = (-S * sci.norm.pdf(d1) * sigma / (2*np.sqrt(T)) - r * K * np.exp(-r*T)* sci.norm.cdf(d2))/365
    return theta
bs_call_theta(100,100,30/365,0.02,0.20)

-0.04079700631509862

In [8]:
# Call Rho per 1% rate = K*T*e^(-rT)*N(d2) / 100
def bs_call_rho(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T)) 
    d2 = d1 - sigma * np.sqrt(T)
    rho = K * T * np.exp(-r*T) * sci.norm.cdf(d2) /100
    return rho
bs_call_rho(100,100,30/365,0.02,0.20)

0.04102839101676596

### 2. Explained PnL & Actual PnL
- Explained PnL = Delta + Gamma + Vega + Theta + Rho
- Actual PnL = (price1 - price0) * qty 


In [9]:
# LONG CALL
K, r, qty = 100, 0.02, 100
S0, S1 = 100, 102
vol0, vol1 = 0.20, 0.21
days_in_year = 365   # calendar-day convention (use 252 for trading days)
T0 = 30/days_in_year #time to maturity in years (30 days)
days = 1
r1 = 0.02

delta = bs_call_delta(S0, K, T0, r, vol0)
gamma = bs_gamma(S0, K, T0, r, vol0)
vega = bs_vega(S0, K, T0, r, vol0)
theta = bs_call_theta(S0, K, T0, r, vol0)
rho = bs_call_rho(S0, K, T0, r, vol0)

# PnL 
delta_PnL = delta * (S1 - S0) * qty
gamma_PnL = 1/2 * gamma * (S1-S0)**2 * qty
vega_PnL = vega * (vol1 - vol0)*100 * qty 
theta_PnL = theta * days * qty
rho_PnL = rho * (r1-r)*100 * qty

print(f"delta = {delta:.4f} | gamma = {gamma:.4f} | vega = {vega:.4f} |"
      f"theta = {theta:.4f} | rho = {rho:.4f}"
      )
print(f"delta_PnL = {delta_PnL: .4f} | gamma_PnL = {gamma_PnL: .4f} | vega_PnL = {vega_PnL: .4f} | "
      f"theta_PnL = {theta_PnL: .4f} | rho_PnL = {rho_PnL: .4f}")

explained = sum([delta_PnL, gamma_PnL, vega_PnL, theta_PnL, rho_PnL])
T1 = T0 - days/days_in_year
price0 = bs_call_prices(S0, K, T0, r, vol0)
price1 = bs_call_prices(S1, K, T1, r1, vol1)
actual = (price1 - price0) * qty 
unexplained = actual - explained
print( f"explained_pnl = {explained: .4f} | actual_pnl = {actual: .4f} | unexplained_pnl = {unexplained: .4f}")


delta = 0.5229 | gamma = 0.0695 | vega = 0.1142 |theta = -0.0408 | rho = 0.0410
delta_PnL =  104.5724 | gamma_PnL =  13.8925 | vega_PnL =  11.4185 | theta_PnL = -4.0797 | rho_PnL =  0.0000
explained_pnl =  125.8038 | actual_pnl =  124.7791 | unexplained_pnl = -1.0248


### 3. Scenario analysis: 
When does the Greeks approximation break down?

Goal: test how well Greeks explain the PnL of the long call across different market moves.

- Wrap the PnL explain into a function `pnl_explain(S1, vol1)`
- Run 5 scenarios (small move, down move, large up move, large down move)
- Compare Explained vs Actual PnL: how does Unexplained change?

In [10]:
def pnl_explained(S1,vol1):
    delta_PnL = delta * (S1 - S0) * qty
    gamma_PnL = 1/2 * gamma * (S1-S0)**2 * qty
    vega_PnL = vega * (vol1 - vol0)*100 * qty 
    theta_PnL = theta * days * qty
    rho_PnL = rho * (r1-r)*100 * qty
    explained = sum([delta_PnL, gamma_PnL, vega_PnL, theta_PnL, rho_PnL])

    T1 = T0 - days/days_in_year
    price1 = bs_call_prices(S1, K, T1, r1, vol1)
    actual = (price1 - price0) * qty
    unexplained = actual - explained
    return explained, actual, unexplained
scenarios = {
    "Base case": (100, 0.20),
    "Spot +2%, Vol +1pt" : (102,0.21),
    "Spot -2%": (98,0.20),
    "Spot +10% (stress up)" : (110,0.20),
    "Spot -10% (stress down)" : (90,0.20)
}
rows = []
for name, (spot,vol) in scenarios.items():
    explained, actual, unexplained = pnl_explained(spot,vol)
    rows.append({
        "Scenario" : name,
        "Explained PnL": explained,
        "Actual PnL" : actual,
        "Unexplained PnL": unexplained
    })
df = pd.DataFrame(rows)
df["Unexplained/Actual (%)"] = df["Unexplained PnL"]/ df["Actual PnL"].abs() *100
df.round(2)


,Scenario,Explained PnL,Actual PnL,Unexplained PnL,Unexplained/Actual (%)
0,Base case,-4.08,-4.11,-0.03,-0.79
1,"Spot +2%, Vol +1pt",125.80,124.78,-1.02,-0.82
2,Spot -2%,-94.76,-94.44,0.32,0.34
3,Spot +10% (stress up),866.10,789.36,-76.74,-9.72
4,Spot -10% (stress down),-179.63,-229.96,-50.33,-21.89


In [11]:
print(bs_gamma(100, K, T0, r, vol0))
print(bs_gamma(110, K, T0, r, vol0))
print(bs_gamma(90,  K, T0, r, vol0))

0.06946274895134844
0.014420353462641883
0.01585142130521138


### Results : 
- Black-Scholes, European call, no dividends, 365-day calendar convention.
- Position: long 100 calls (K = 100, S0 = 100, 30 days to maturity), 1-day horizon.

| Scenario | Explained | Actual | Unexplained | Unexplained / Actual |
|---|---|---|---|---|
| Base case (time only) | -4.08 | -4.11 | -0.03 | 0.7% |
| Spot +2%, Vol +1pt | 125.80 | 124.78 | -1.02 | 0.8% |
| Spot -2% | -94.76 | -94.44 | 0.32 | 0.3% |
| Spot +10% (stress up) | 866.10 | 789.36 | -76.74 | 9.7% |
| Spot -10% (stress down) | -179.63 | -229.96 | -50.33 | 21.9% |

### Observations
- **Small moves (up to +/- 2%):** Greeks explain the PnL almost fully (Unexplained below 1% of Actual). In the base case, the loss is entirely Theta (time decay).
- **Large moves (+/-10%):** Unexplained grows disproportionately (about 10% to 22% of Actual)
- Greeks are local approximation at S0. Gamma peaks ATM (0.0695 at S =100) and falls sharply away from the strike (0.0144 at S = 110, 0.0159 at S = 90). 
- The Gamma PnL term assumes it stays constant (fixed at S0), but in reality Gamma changes as spot moves. The approximation therefore overstates the Gamma gain on both sides.
- Greeks-based estimates are too optimistic for large moves: they overstate the gain in the stress-up scenario (explained = 866 vs actual = 789) and understate the loss in the stress-down scenario (explained = -180 vs actual = -230). Stress tests and tail risk measures should therefore use full revaluation.


# Part II: Portfolio PnL Explain

### 4. Put option (Black-Scholes)

Same d1,d2 as the call
- Price = K * e^(-rT)* N(-d2) - S * N(-d1)
- Delta: N(d1) - 1 (negative)
- Gamma, Vega : identical to the call
- Check : put-call parity, C - P = S - K * e^(-rT)

In [12]:

# d1 = [ln(S/K) + (r + σ²/2)* T] / (σ√T)
# d2 = d1 − σ√T
# Put price = K * e^(-rT)* N(-d2) - S * N(-d1)
def bs_put_prices(S,K,T,r,sigma):
    d1 = (np.log(S/K)+ (r + sigma**2/2)*T)/ (sigma * np.sqrt(T))
    d2 = d1 - sigma* np.sqrt(T)
    price = K * np.exp(-r*T) * sci.norm.cdf(-d2) - S * sci.norm.cdf(-d1)
    return price
bs_put_prices(100,100,30/365,0.02,0.2)

2.2040868906830653

In [13]:

def bs_put_delta(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    delta = sci.norm.cdf(d1) - 1 
    return delta
bs_put_delta (100,100,30/365, 0.02, 0.20)

def bs_put_theta(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    theta = (-S * sci.norm.pdf(d1) * sigma / (2*np.sqrt(T)) + r * K * np.exp(-r*T)* sci.norm.cdf(-d2))/365
    return theta
bs_put_theta(100,100,30/365,0.02,0.20)

def bs_put_rho(S,K,T,r,sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T)/(sigma * np.sqrt(T)) 
    d2 = d1 - sigma * np.sqrt(T)
    rho = -K * T * np.exp(-r*T) * sci.norm.cdf(-d2) /100
    return rho
print(f"delta_put = {bs_put_delta(100,100,30/365,0.02,0.20): .4f}")
print(f"theta_put = {bs_put_theta(100,100,30/365,0.02,0.20): .4f}")
print(f"rho_put = {bs_put_rho(100,100,30/365,0.02,0.20): .4f}")

delta_put = -0.4771
theta_put = -0.0353
rho_put = -0.0410


### 5. Portfolio & Scenario definition

Signed quantity convention: long = positive, short = negative.
Greeks are computed per 1 long option, then multiplied by `qty`.

| Position | Type | Strike | Qty |
|---|---|---|---|
| Stock | stock | - | +100 |
| Long Call 100 | call | 100 | +100 |
| Long Put 95 | put | 95 | +100 |
| Short Call 105 | call | 105 | -50 |

In [44]:
# Base market state (day 0)
S0, vol0, r = 100, 0.20, 0.02
days_in_year = 365
T0 = 30 / days_in_year

In [ ]:
positions = [
    {"name": "Stock",       "type": "stock", "K": None, "qty": 100},
    {"name": "Long Call 100", "type": "call",  "K": 100,  "qty": 100},
    {"name": "Long Put 95",   "type": "put",   "K": 95,   "qty": 100},
    {"name": "Short Call 105","type": "call",  "K": 105,  "qty": -50},
]
pd.DataFrame(positions)

,name,type,K,qty
0,Stock,stock,NaN,100
1,Long Call 100,call,100.0,100
2,Long Put 95,put,95.0,100
3,Short Call 105,call,105.0,-50


In [39]:
scenarios = [
    {"name": "Base case (time only)", "S1": 100, "vol1": 0.20, "days": 1},
    {"name": "Spot +2%, Vol +1pt",    "S1": 102, "vol1": 0.21, "days": 1},
    {"name": "Spot -2%",              "S1": 98,  "vol1": 0.20, "days": 1},
    {"name": "Spot +10% (stress up)", "S1": 110, "vol1": 0.20, "days": 1},
    {"name": "Spot -10% (stress down)", "S1": 90, "vol1": 0.20, "days": 1},
]
pd.DataFrame(scenarios)

,name,S1,vol1,days
0,Base case (time only),100,0.20,1
1,"Spot +2%, Vol +1pt",102,0.21,1
2,Spot -2%,98,0.20,1
3,Spot +10% (stress up),110,0.20,1
4,Spot -10% (stress down),90,0.20,1


### 6. Position-level Greeks
Portfolio Greeks = sum of position Greeks.


In [42]:
def position_greeks(pos,S,T,r,sigma):
    K, qty = pos["K"], pos["qty"]
    if pos["type"] == "stock":
        delta, gamma, vega, theta, rho = 1,0,0,0,0
    elif pos["type"] == "call":
        delta = bs_call_delta(S,K,T,r,sigma)
        gamma = bs_gamma (S,K,T,r,sigma)
        vega = bs_vega (S,K,T,r,sigma)
        theta = bs_call_theta (S,K,T,r,sigma)
        rho = bs_call_rho (S,K,T,r,sigma)
    elif pos["type"] =="put":
        delta = bs_put_delta(S,K,T,r,sigma)
        gamma = bs_gamma (S,K,T,r,sigma)
        vega = bs_vega (S,K,T,r,sigma)
        theta = bs_put_theta (S,K,T,r,sigma)
        rho = bs_put_rho (S,K,T,r,sigma)
    return {"delta": delta * qty, "gamma": gamma*qty, "vega": vega*qty, "theta": theta*qty, "rho": rho*qty}
print(position_greeks(positions[3], 100, 30/365, 0.02, 0.20)) #Short Call 105 


{'delta': -10.68599187706426, 'gamma': -2.539118977599191, 'vega': -4.173894209752095, 'theta': 1.4479931819134892, 'rho': -0.8504266799418662}


In [43]:
rows = []
for pos in positions:
    greek = position_greeks(pos, S0, T0, r, vol0)
    rows.append({"Position": pos["name"], "Delta": greek["delta"], "Gamma": greek["gamma"], "Vega": greek["vega"], "Theta": greek["theta"], "Rho": greek["rho"]})
greeks_df = pd.DataFrame(rows)
greeks_df

,Position,Delta,Gamma,Vega,Theta,Rho
0,Stock,100.000000,0.000000,0.000000,0.000000,0.000000
1,Long Call 100,52.286211,6.946275,11.418534,-4.079701,4.102839
2,Long Put 95,-17.057068,4.422834,7.270412,-2.327064,-1.446103
3,Short Call 105,-10.685992,-2.539119,-4.173894,1.447993,-0.850427


### 7. Portfolio Greeks
- Portfolio Greeks = sum of the position Greeks (already multiplied by quantity)
- Delta, Gamma: in equivalent shares
-Vega: € per vol point, Theta: € per day, Rho: € per 1% rate move

In [36]:
ptf_greeks = greeks_df.sum(numeric_only=True)
ptf_greeks

Delta    124.543151
Gamma      8.829990
Vega      14.515052
Theta     -4.958771
Rho        1.806309
dtype: float64

### 8. Portfolio PnL (Explained, Actual, Unexplained)

In [45]:
pd.DataFrame(scenarios)

,name,S1,vol1,days
0,Base case (time only),100,0.20,1
1,"Spot +2%, Vol +1pt",102,0.21,1
2,Spot -2%,98,0.20,1
3,Spot +10% (stress up),110,0.20,1
4,Spot -10% (stress down),90,0.20,1


In [46]:
pd.DataFrame(positions)

,name,type,K,qty
0,Stock,stock,NaN,100
1,Long Call 100,call,100.0,100
2,Long Put 95,put,95.0,100
3,Short Call 105,call,105.0,-50


In [67]:
# Explained PnL 
def ptf_pnl_explained (ptf_greeks, S0, S1, vol0,vol1,r0,r1,days) :
    delta_PnL = (S1-S0) * ptf_greeks["Delta"]
    gamma_PnL = 1/2 * ptf_greeks["Gamma"] * (S1-S0)**2
    vega_PnL = ptf_greeks["Vega"] * (vol1 - vol0)*100
    theta_PnL = ptf_greeks["Theta"] * days 
    rho_PnL = ptf_greeks["Rho"] * (r1-r0)*100
    return sum([delta_PnL, gamma_PnL, vega_PnL, theta_PnL, rho_PnL])

for scenario in scenarios:
    explained = ptf_pnl_explained(ptf_greeks, S0, scenario["S1"], vol0, scenario["vol1"], r, r, scenario["days"])
    print(f"Portfolio Explained PnL ({scenario['name']}): {explained:.2f}")

# Actual PnL
# price
def position_price(pos,S,T,r,sigma) :
    K = pos["K"]
    if pos["type"] == "stock":
        price = S
    elif pos["type"] == "call":
        price = bs_call_prices(S,K,T,r,sigma)
    else:
        price = bs_put_prices(S,K,T,r,sigma)
    return price

# Price by scenario
rows = []
for scenario in scenarios :
    T1 = T0 - scenario["days"]/365
    actual_pnl = 0
    for position in positions:
        price0 = position_price(position, S0, T0, r, vol0)
        price1 = position_price(position, scenario["S1"], T1, r, scenario["vol1"])
        actual_pnl += (price1 - price0) * position["qty"]
    print(f"Portfolio Actual PnL ({scenario['name']}): {actual_pnl:.2f}")
# Unexplained PnL
    explained = ptf_pnl_explained(ptf_greeks, S0, scenario["S1"], vol0, scenario["vol1"], r, r, scenario["days"])
    actual = actual_pnl
    unexplained = actual - explained
    rows.append({"Scenario" : scenario["name"],
                 "Explained PnL": explained,
                 "Actual PnL": actual,
                 "Unexplained PnL": unexplained,
    })
results_df = pd.DataFrame(rows)
results_df["Unexplained / Actual (%)"] = results_df["Unexplained PnL"].abs()/results_df["Actual PnL"].abs() * 100
results_df.round(2)

Portfolio Explained PnL (Base case (time only)): -4.96
Portfolio Explained PnL (Spot +2%, Vol +1pt): 276.30
Portfolio Explained PnL (Spot -2%): -236.39
Portfolio Explained PnL (Spot +10% (stress up)): 1681.97
Portfolio Explained PnL (Spot -10% (stress down)): -808.89
Portfolio Actual PnL (Base case (time only)): -4.99
Portfolio Actual PnL (Spot +2%, Vol +1pt): 271.70
Portfolio Actual PnL (Spot -2%): -235.69
Portfolio Actual PnL (Spot +10% (stress up)): 1478.91
Portfolio Actual PnL (Spot -10% (stress down)): -715.73


,Scenario,Explained PnL,Actual PnL,Unexplained PnL,Unexplained / Actual (%)
0,Base case (time only),-4.96,-4.99,-0.03,0.70
1,"Spot +2%, Vol +1pt",276.30,271.70,-4.60,1.69
2,Spot -2%,-236.39,-235.69,0.70,0.30
3,Spot +10% (stress up),1681.97,1478.91,-203.06,13.73
4,Spot -10% (stress down),-808.89,-715.73,93.17,13.02


# Actual PnL
